In [1]:
from __future__ import annotations

import numpy as np
import netCDF4
import yaml
import xarray as xr
from modular_dales import dales_simulation
from modular_dales.Atmosphere import (
    AtmosphereModule,
    AtmosphericProfile,
    InterpolatedProfile,
)
import subprocess
from modular_dales.Configuration.defaultnamelist import DefaultNamelistModule
from modular_dales.Configuration.output_modules import EasyOutputModule
from modular_dales.Configuration.run_and_time import TimeModule
from modular_dales.Geometry.GridDales import GridDales
from modular_dales.modular.time_dependent import (
    TimeDependentScalar,
    TimedependentModule,
)
from modular_dales.Surface.surface import ConstantSurfaceTemperatureModule
from modular_dales.IBM import IBMModule, FromAHN
from modular_dales.vars import *  # noqa: F401,F403

with open("machine_conf.yaml", "r") as file:
    machine_conf = yaml.safe_load(file)

Roughness element should be at least 1m for MOST stability!!! Setting to 1m
Roughness element should be at least 1m for MOST stability!!! Setting to 1m
Roughness element should be at least 1m for MOST stability!!! Setting to 1m


In [2]:
"""Construct a simulation with time-dependent scalar params inside AtmosphericProfile."""
sim = dales_simulation("timedep_atmosphere_params", machine_conf)
sim += DefaultNamelistModule()
domain_info = GridDales(
    itot=16,
    jtot=16,
    kmax=160,
    xsize=160.0,
    ysize=160.0,
    kmax_soil=4,
    xlat=52.25,
    xlon=5.45,
    x0=137167.763496,
    y0=454496.628974,
    alpha=1.01,
    dz0=5,
    proj4="+proj=sterea +lat_0=52.1561605555556 +lon_0=5.38763888888889 +k=0.9999079 +x_0=155000 +y_0=463000 +ellps=bessel +towgs84=565.4171,50.3319,465.5524,1.9342,-1.6677,9.1019,4.0725 +units=m +no_defs +type=crs",
)
sim += domain_info


timesteps = [0,3600,7200,14400]
sim += TimedependentModule(timesteps=timesteps)


atmo = AtmosphereModule()

atmo += AtmosphericProfile(
    variable=ua, shape="lin", params=dict(surf_val=0, ddz=1e-3)
)
atmo += AtmosphericProfile(
    variable=va, shape="lin", params=dict(surf_val=0, ddz=0))

atmo += AtmosphericProfile(
    variable=thetal, shape="lin", params=dict(surf_val=293.15, ddz=5e-5)
)
atmo += AtmosphericProfile(
    variable=qt, shape="lin", params=dict(surf_val=0, ddz=0))

atmo += InterpolatedProfile(
    variable=tke,
    z=[0, 4000, 5000],
    points=[1, 1e-8, 1e-8],
)
sim += atmo


sim += ConstantSurfaceTemperatureModule(
    thls=TimeDependentScalar(
        times=timesteps,
        values=[293.15, 293.15, 300.15,300.15]
    ),
    z0mav=0.0001,
    z0hav=0.0001,
    ps=100000,
    albedoav=0.22,
)

# ibm = IBMModule()
# ibm += FromAHN()
# sim += ibm

                  


sim += TimeModule(xtime=0.0, xday=1, xyear=2025, runtime=14400)
output_dict = {
                    "namfielddump:":60,
                "namcape":10,
                "namlsmcrosssection:dtav":60,
                "namgenstat:dtav":10,
                "namcrosssection:dtav":10,
                "namtimestat:dtav":10,
                "nambudget:dtav":10,
                "nambudget:timeav":10,
                "namgenstat:timeav":10,
}
sim += EasyOutputModule(output_interval=10)

if sim.nml.get("namchecksim") is None:
    sim.nml["namchecksim"] = {}
sim.nml["namchecksim"]["tcheck"] = 360

sim.sim_preprocessing_pipeline()
print(sim.output_path)

/Users/andrevanginkel/Documents/40_Input_and_Runs/42_Dales_Cases/42.01_generated_cases/timedep_atmosphere_params


In [2]:
ds = None

subprocess.run(["rm","-rf", "run_001"],cwd=sim.output_path.as_posix())
subprocess.run(["rm","fielddump.nc","cape.nc","surfcross.nc"],cwd=sim.output_path.as_posix())
subprocess.run("./job.001", cwd=sim.output_path.as_posix())

subprocess.run(
    ["combine.sh", "run_001"], check=False, cwd=sim.output_path.as_posix())


NameError: name 'sim' is not defined

In [ ]:
import panel as pn
import hvplot.xarray  # noqa: F401
import pathlib
import dataclasses

ds_fielddump = xr.open_dataset(sim.output_path / "fielddump.nc") 


ds_fielddump.thl.mean(dim=("xt","yt")).plot(x="time")

In [10]:
# ds_fielddump.w.std(dim=("xt","yt")).plot(x="time")

In [11]:
tm = sim.output_path/"run_001"/"profiles.001.nc"
ds_tm = xr.open_dataset(tm)
for var in ds_tm:
    print(var, ds_tm[var].long_name)

rhof Full level slab averaged density
rhobf Full level base-state density
rhobh Half level base-state density
presh Pressure at cell center
u West-East velocity
v South-North velocity
w Vertical velocity
thl Liquid water potential temperature
thv Virtual potential temperature
qt Total water specific humidity
ql Liquid water specific humidity
wthls SFS-Theta_l flux
wthlr Resolved Theta_l flux
wthlt Total Theta_l flux
wthvs SFS-buoyancy flux
wthvr Resolved buoyancy flux
wthvt Total buoyancy flux
wqts SFS-moisture flux
wqtr Resolved moisture flux
wqtt Total moisture flux
wqls SFS-liquid water flux
wqlr Resolved liquid water flux
wqlt Total liquid water flux
uws SFS-momentum flux (uw)
uwr Resolved momentum flux (uw)
uwt Total momentum flux (uw)
vws SFS-momentum flux (vw)
vwr Resolved momentum flux (vw)
vwt Total momentum flux (vw)
w2s SFS-TKE
w2r Resolved vertical velocity variance
skew vertical velocity skewness
u2r Resolved horizontal velocity variance (u)
v2r Resolved horizontal velocit

In [14]:
import hvplot.xarray
ds_sel = ds_tm#.sel(zt=slice(0,800),zm=slice(0,800))
ds = ds_sel.interp(zt=np.linspace(ds_sel.zt.min().values,ds_sel.zt.max().values,num=40),zm=np.linspace(ds_sel.zm.min().values,ds_sel.zm.max().values,num=40))

In [12]:
import holoviews as hv
import hvplot.xarray  # noqa

hv.extension("bokeh")
ds_sel = ds_tm.sel(zt=slice(0,800),zm=slice(0,800))
ds = ds_sel.interp(zt=np.linspace(ds_sel.zt.min().values,ds_sel.zt.max().values,num=90),zm=np.linspace(ds_sel.zm.min().values,ds_sel.zm.max().values,num=90))

# Build label mapping: "short: long" -> short
label_to_var = {}

for v in ds.data_vars:
    long_name = ds[v].attrs.get("long_name", "")
    if long_name:
        label = f"{v}: {long_name}"
    else:
        label = v
    label_to_var[label] = v

variable_dim = hv.Dimension(
    "variable",
    values=list(label_to_var.keys()),
    label="Variable",
)

def plot_variable(label):
    var = label_to_var[label]
    da = ds[var]

    # detect vertical coordinate automatically
    if "zt" in da.dims:
        ydim = "zt"
    elif "zm" in da.dims:
        ydim = "zm"
    else:
        raise ValueError(f"{var} has no vertical dimension")
    # compute min/max safely (lazy compatible)
    vmin = float(da.min())
    vmax = float(da.max())
    # decide symmetric scaling
    if vmin < 0 and vmax > 0:
        vmax_abs = max(abs(vmin), abs(vmax))
        clim = (-vmax_abs, vmax_abs)
        cmap = "RdBu_r"   # diverging colormap
    else:
        clim = (vmin, vmax)
        cmap = "viridis"  # sequential colormap

    return da.hvplot(
        x="time",
        y=ydim,
        cmap=cmap,
        clim=clim,
        colorbar=True,
        title=label,
    )

hv.DynamicMap(plot_variable, kdims=[variable_dim])

:DynamicMap   [variable]
   :Image   [time,zt]   (Full level slab averaged density)